In [ ]:

# Install all required libraries
!pip -q install -U yt-dlp
!pip -q install -U openai-whisper
!pip -q install -U google-generativeai
!pip -q install -U pandas
!pip -q install -U ffmpeg-python

# Install FFmpeg
!apt-get -qq update
!apt-get -qq install ffmpeg

print("Libraries Installed Successfully")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 50.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 11.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.7/197.7 MB 6.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 351.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 78.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy 

In [ ]:
import os
import json
import subprocess

import yt_dlp
import whisper
import pandas as pd

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [ ]:
folders = [
    "downloads",
    "audio",
    "transcripts",
    "highlights"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created!")

print("\nFolder Structure:")
for folder in folders:
    print("📁", folder)

Project folders created!

Folder Structure:
📁 downloads
📁 audio
📁 transcripts
📁 highlights


In [ ]:
youtube_url = input("🎥 Enter YouTube URL: ").strip()

print("\nYou entered:")
print(youtube_url)

🎥 Enter YouTube URL: https://www.youtube.com/watch?v=eVFzbxmKNUw

You entered:
https://www.youtube.com/watch?v=eVFzbxmKNUw


In [ ]:
import yt_dlp
import os

def download_video(youtube_url):

    print("\nStarting download...\n")

    ydl_opts = {
        "format": "bestvideo+bestaudio/best",
        "merge_output_format": "mp4",
        "outtmpl": "downloads/video.%(ext)s",
        "noplaylist": True,
        "quiet": False,
        "geo_bypass": True,
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([youtube_url])

        print("\n Download completed!")

    except Exception as e:
        print("\n Download failed!")
        print(e)

In [ ]:
download_video(youtube_url)


Starting download...

[youtube] Extracting URL: https://www.youtube.com/watch?v=eVFzbxmKNUw
[youtube] eVFzbxmKNUw: Downloading webpage


[youtube] eVFzbxmKNUw: Downloading android vr player API JSON
[info] eVFzbxmKNUw: Downloading 1 format(s): 401+251
[download] Destination: downloads/video.f401.mp4
[download] 100% of  221.34MiB in 00:00:04 at 48.62MiB/s  
[download] Destination: downloads/video.f251.webm
[download] 100% of   11.46MiB in 00:00:00 at 48.37MiB/s  
[Merger] Merging formats into "downloads/video.mp4"
Deleting original file downloads/video.f401.mp4 (pass -k to keep)
Deleting original file downloads/video.f251.webm (pass -k to keep)

 Download completed!


In [ ]:
import os

video_path = None

for file in os.listdir("downloads"):
    if file.endswith((".mp4", ".mkv", ".webm", ".mov")):
        video_path = os.path.join("downloads", file)
        break

if video_path:
    print("Video Found:")
    print(video_path)
else:
    print("No video found!")

Video Found:
downloads/video.mp4


In [ ]:
audio_path = "audio/audio.wav"

command = [
    "ffmpeg",
    "-i",
    video_path,
    "-ar",
    "16000",
    "-ac",
    "1",
    audio_path,
    "-y"
]

subprocess.run(command)

print("Audio Extracted Successfully!")
print(audio_path)

Audio Extracted Successfully!
audio/audio.wav


In [ ]:
import os

if os.path.exists(audio_path):
    print("Audio file created successfully!")
else:
    print("Audio extraction failed.")

Audio file created successfully!


In [ ]:
import whisper

print("Loading Whisper model...")

# Choose one:
# tiny   -> Fastest, least accurate
# base   -> Good for testing
# small  -> Better accuracy
# medium -> Even better (requires more GPU memory)

model = whisper.load_model("base")

print("Whisper model loaded successfully!")

Loading Whisper model...


100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 153MiB/s]


Whisper model loaded successfully!


In [ ]:
print("Transcribing audio...")

result = model.transcribe(
    audio_path,
    language="en"
)

segments = result["segments"]

print(f"Transcription completed!")
print(f"Total segments: {len(segments)}")

Transcribing audio...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription completed!
Total segments: 293


In [ ]:

import pandas as pd

transcript = []

for seg in segments:
    transcript.append({
        "Start": seg["start"],
        "End": seg["end"],
        "Text": seg["text"].strip()
    })

df = pd.DataFrame(transcript)

df.head(10)

,Start,End,Text
0,0.00,7.44,"[♪ Music playing in background, applause and c..."
1,7.44,9.34,Picture this.
2,9.34,12.08,You're going on a boat trip.
3,12.08,14.48,"And you get on board with your family,"
4,14.48,16.08,and you got your bags.
5,16.08,19.58,"And the captain comes out to greet you and says,"
6,19.58,21.08,Hi.
7,21.08,22.76,My name's Montana.
8,22.76,24.32,"Uh, bonfless."
9,24.32,28.80,"Oh, so I'll be your captain for this journey."


In [ ]:
import pandas as pd

rows = []

for seg in segments:

    rows.append({

        "start": float(seg["start"]),
        "end": float(seg["end"]),
        "duration": round(seg["end"]-seg["start"],2),
        "text": seg["text"]

    })

df = pd.DataFrame(rows)

print("Total Segments :",len(df))

df.head()

Total Segments : 293


,start,end,duration,text
0,0.00,7.44,7.44,"[♪ Music playing in background, applause and ..."
1,7.44,9.34,1.90,Picture this.
2,9.34,12.08,2.74,You're going on a boat trip.
3,12.08,14.48,2.40,"And you get on board with your family,"
4,14.48,16.08,1.60,and you got your bags.


In [ ]:
import re

def clean_text(text):

    text = str(text)

    text = text.replace("\n"," ")

    text = re.sub(r"\s+"," ",text)

    text = text.strip()

    return text

df["clean_text"] = df["text"].apply(clean_text)

df.head()

,start,end,duration,text,clean_text
0,0.00,7.44,7.44,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c..."
1,7.44,9.34,1.90,Picture this.,Picture this.
2,9.34,12.08,2.74,You're going on a boat trip.,You're going on a boat trip.
3,12.08,14.48,2.40,"And you get on board with your family,","And you get on board with your family,"
4,14.48,16.08,1.60,and you got your bags.,and you got your bags.


In [ ]:

before = len(df)

df = df[df["clean_text"]!=""]

after = len(df)

print("Removed :",before-after)
print("Remaining :",after)

Removed : 0
Remaining : 293


In [ ]:
df["word_count"] = df["clean_text"].apply(lambda x: len(x.split()))

df = df[df["word_count"]>=3]

print("Remaining Segments :",len(df))

Remaining Segments : 284


In [ ]:
before = len(df)

df = df.drop_duplicates(subset="clean_text")

after = len(df)

print("Duplicates Removed :",before-after)

Duplicates Removed : 0


In [ ]:
df = df.reset_index(drop=True)

df.head()

,start,end,duration,text,clean_text,word_count
0,0.00,7.44,7.44,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c...",8
1,9.34,12.08,2.74,You're going on a boat trip.,You're going on a boat trip.,6
2,12.08,14.48,2.40,"And you get on board with your family,","And you get on board with your family,",8
3,14.48,16.08,1.60,and you got your bags.,and you got your bags.,5
4,16.08,19.58,3.50,"And the captain comes out to greet you and says,","And the captain comes out to greet you and says,",10


In [ ]:
merged = []

i = 0

while i < len(df):

    current = df.iloc[i].copy()

    while (

        i+1 < len(df)

        and current["word_count"]<8

    ):

        nxt = df.iloc[i+1]

        current["clean_text"] += " " + nxt["clean_text"]

        current["end"] = nxt["end"]

        current["duration"] = round(

            current["end"]-current["start"],2

        )

        current["word_count"] = len(

            current["clean_text"].split()

        )

        i += 1

    merged.append(current)

    i += 1

merged_df = pd.DataFrame(merged)

print("Merged Segments :",len(merged_df))

Merged Segments : 189


In [ ]:
def timestamp(sec):

    m = int(sec//60)

    s = int(sec%60)

    return f"{m:02d}:{s:02d}"

merged_df["timestamp"] = merged_df["start"].apply(timestamp)

merged_df.head()

,start,end,duration,text,clean_text,word_count,timestamp
0,0.00,7.44,7.44,"[♪ Music playing in background, applause and ...","[♪ Music playing in background, applause and c...",8,00:00
1,9.34,14.48,5.14,You're going on a boat trip.,You're going on a boat trip. And you get on bo...,14,00:09
3,14.48,19.58,5.10,and you got your bags.,and you got your bags. And the captain comes o...,15,00:14
5,21.08,28.80,7.72,My name's Montana.,"My name's Montana. Oh, so I'll be your captain...",12,00:21
7,29.16,33.60,4.44,"So, uh, oh boy.","So, uh, oh boy. Let's just have a great trip.",10,00:29


In [ ]:
merged_df.to_csv(

    "transcripts/clean_transcript.csv",

    index=False

)

print("Saved Successfully")

Saved Successfully


In [ ]:
merged_df[
    [
        "timestamp",
        "clean_text",
        "word_count"
    ]
].head(20)

,timestamp,clean_text,word_count
0,00:00,"[♪ Music playing in background, applause and c...",8
1,00:09,You're going on a boat trip. And you get on bo...,14
3,00:14,and you got your bags. And the captain comes o...,15
5,00:21,"My name's Montana. Oh, so I'll be your captain...",12
7,00:29,"So, uh, oh boy. Let's just have a great trip.",10
9,00:34,[♪ Laughter and laughter. Get me off of this b...,10
11,00:40,What we want in that moment is for the captain...,13
12,00:44,"and say, Hi. My name is Montana Bonfless.",8
14,00:47,I'll be your captain for this journey. Let's h...,12
16,00:52,"The point is, when you are the speaker,",8


In [ ]:
conversation_data = []

for i in range(len(merged_df)):

    previous_text = ""
    current_text = merged_df.iloc[i]["clean_text"]
    next_text = ""

    if i > 0:
        previous_text = merged_df.iloc[i-1]["clean_text"]

    if i < len(merged_df)-1:
        next_text = merged_df.iloc[i+1]["clean_text"]

    conversation = f"""
Previous:
{previous_text}

Current:
{current_text}

Next:
{next_text}
"""

    conversation_data.append({
        "start": merged_df.iloc[i]["start"],
        "end": merged_df.iloc[i]["end"],
        "timestamp": merged_df.iloc[i]["timestamp"],
        "conversation": conversation
    })

conversation_df = pd.DataFrame(conversation_data)

print("Conversation windows created:", len(conversation_df))

Conversation windows created: 189


In [ ]:
conversation_df.head(5)

,start,end,timestamp,conversation
0,0.00,7.44,00:00,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...
1,9.34,14.48,00:09,"\nPrevious:\n[♪ Music playing in background, a..."
2,14.48,19.58,00:14,\nPrevious:\nYou're going on a boat trip. And ...
3,21.08,28.80,00:21,\nPrevious:\nand you got your bags. And the ca...
4,29.16,33.60,00:29,"\nPrevious:\nMy name's Montana. Oh, so I'll be..."


In [ ]:
print(conversation_df.loc[5, "conversation"])


Previous:
So, uh, oh boy. Let's just have a great trip.

Current:
[♪ Laughter and laughter. Get me off of this boat.

Next:
What we want in that moment is for the captain to walk out



In [ ]:
def count_words(text):
    return len(text.split())

def question_mark(text):
    return int("?" in text)

conversation_df["word_count"] = merged_df["clean_text"].apply(count_words)

conversation_df["contains_question"] = merged_df["clean_text"].apply(question_mark)

conversation_df["duration"] = merged_df["duration"]

conversation_df.head()

,start,end,timestamp,conversation,word_count,contains_question,duration
0,0.00,7.44,00:00,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...,8.0,0.0,7.44
1,9.34,14.48,00:09,"\nPrevious:\n[♪ Music playing in background, a...",14.0,0.0,5.14
2,14.48,19.58,00:14,\nPrevious:\nYou're going on a boat trip. And ...,NaN,NaN,NaN
3,21.08,28.80,00:21,\nPrevious:\nand you got your bags. And the ca...,15.0,0.0,5.10
4,29.16,33.60,00:29,"\nPrevious:\nMy name's Montana. Oh, so I'll be...",NaN,NaN,NaN


In [ ]:
agreement_words = [
    "yes",
    "yeah",
    "exactly",
    "correct",
    "right",
    "absolutely",
    "true",
    "definitely",
    "agree"
]

def agreement_score(text):

    text = text.lower()

    score = 0

    for word in agreement_words:
        if word in text:
            score += 1

    return score

conversation_df["agreement_score"] = merged_df["clean_text"].apply(agreement_score)

In [ ]:
disagreement_words = [
    "no",
    "not",
    "never",
    "don't",
    "disagree",
    "wrong",
    "however",
    "actually",
    "but"
]

def disagreement_score(text):

    text = text.lower()

    score = 0

    for word in disagreement_words:
        if word in text:
            score += 1

    return score

conversation_df["disagreement_score"] = merged_df["clean_text"].apply(disagreement_score)

In [ ]:
question_words = [
    "what",
    "why",
    "when",
    "where",
    "who",
    "which",
    "how",
    "can",
    "could",
    "should",
    "would",
    "do",
    "does",
    "did"
]

def question_score(text):

    text = text.lower()

    score = 0

    for word in question_words:
        if word in text:
            score += 1

    if "?" in text:
        score += 2

    return score

conversation_df["question_score"] = merged_df["clean_text"].apply(question_score)

In [ ]:
conversation_df[
    [
        "timestamp",
        "word_count",
        "contains_question",
        "question_score",
        "agreement_score",
        "disagreement_score"
    ]
].head(20)

,timestamp,word_count,contains_question,question_score,agreement_score,disagreement_score
0,00:00,8.0,0.0,0.0,0.0,0.0
1,00:09,14.0,0.0,0.0,0.0,0.0
2,00:14,NaN,NaN,NaN,NaN,NaN
3,00:21,15.0,0.0,0.0,0.0,0.0
4,00:29,NaN,NaN,NaN,NaN,NaN
5,00:34,12.0,0.0,0.0,0.0,0.0
6,00:40,NaN,NaN,NaN,NaN,NaN
7,00:44,10.0,0.0,0.0,0.0,0.0
8,00:47,NaN,NaN,NaN,NaN,NaN
9,00:52,10.0,0.0,0.0,0.0,0.0


In [ ]:
conversation_df.to_csv(
    "transcripts/conversation_features.csv",
    index=False
)

print("Feature dataset saved!")

Feature dataset saved!


In [ ]:
!pip -q install scikit-learn

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

print("TF-IDF Ready")

TF-IDF Ready


In [ ]:
vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=30,
    ngram_range=(1,2)
)

tfidf_matrix = vectorizer.fit_transform(
    conversation_df["conversation"]
)

feature_names = vectorizer.get_feature_names_out()

print("Top Features Extracted:", len(feature_names))

Top Features Extracted: 30


In [ ]:
print("Top Keywords:\n")

for word in feature_names:
    print(word)

Top Keywords:

audience
better
captain
confident
current
day
don
feel
just
know
let
like
loud
make
nervous
new
pause
practice
previous
purpose
really
say
sentence
shape
silent
silent sentence
stage
superhero
time
want


In [ ]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,1.414214
1,00:09,1.517023
2,00:14,1.305217
3,00:21,1.840456
4,00:29,2.034549


In [ ]:

tfidf_scores = np.asarray(tfidf_matrix.sum(axis=1)).flatten()

conversation_df["tfidf_score"] = tfidf_scores

conversation_df[
    [
        "timestamp",
        "tfidf_score"
    ]
].head()

,timestamp,tfidf_score
0,00:00,1.414214
1,00:09,1.517023
2,00:14,1.305217
3,00:21,1.840456
4,00:29,2.034549


In [ ]:
conversation_df[
    [
        "timestamp",
        "tfidf_score",
        "conversation"
    ]
].head(10)

,timestamp,tfidf_score,conversation
0,00:00,1.414214,\nPrevious:\n\n\nCurrent:\n[♪ Music playing in...
1,00:09,1.517023,"\nPrevious:\n[♪ Music playing in background, a..."
2,00:14,1.305217,\nPrevious:\nYou're going on a boat trip. And ...
3,00:21,1.840456,\nPrevious:\nand you got your bags. And the ca...
4,00:29,2.034549,"\nPrevious:\nMy name's Montana. Oh, so I'll be..."
5,00:34,2.261951,"\nPrevious:\nSo, uh, oh boy. Let's just have a..."
6,00:40,2.045444,\nPrevious:\n[♪ Laughter and laughter. Get me ...
7,00:44,2.123399,\nPrevious:\nWhat we want in that moment is fo...
8,00:47,2.036540,"\nPrevious:\nand say, Hi. My name is Montana B..."
9,00:52,1.896224,\nPrevious:\nI'll be your captain for this jou...


In [ ]:
!pip -q install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 4.1 MB/s eta 0:00:00


In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    return abs(analyzer.polarity_scores(text)["compound"])

conversation_df["sentiment_score"] = conversation_df["conversation"].apply(get_sentiment)

conversation_df[["timestamp","sentiment_score"]].head()

,timestamp,sentiment_score
0,00:00,0.7845
1,00:09,0.8481
2,00:14,0.3182
3,00:21,0.7506
4,00:29,0.8885


In [ ]:
features = [

    "question_score",
    "agreement_score",
    "disagreement_score",
    "word_count",
    "duration",
    "sentiment_score",
    "tfidf_score"

]

for feature in features:

    max_val = conversation_df[feature].max()

    if max_val > 0:

        conversation_df[feature] = conversation_df[feature] / max_val

print("Normalization Complete")

Normalization Complete


In [ ]:

WEIGHTS = {

    "question" : 0.25,
    "agreement" : 0.15,
    "disagreement" : 0.15,
    "tfidf" : 0.20,
    "sentiment" : 0.15,
    "word" : 0.05,
    "duration" : 0.05

}

In [ ]:
conversation_df["highlight_score"] = (

WEIGHTS["question"] * conversation_df["question_score"]

+

WEIGHTS["agreement"] * conversation_df["agreement_score"]

+

WEIGHTS["disagreement"] * conversation_df["disagreement_score"]

+

WEIGHTS["tfidf"] * conversation_df["tfidf_score"]

+

WEIGHTS["sentiment"] * conversation_df["sentiment_score"]

+

WEIGHTS["word"] * conversation_df["word_count"]

+

WEIGHTS["duration"] * conversation_df["duration"]

)

In [ ]:
conversation_df["highlight_score"] = (

conversation_df["highlight_score"] * 100

).round(2)

In [ ]:
def classify(row):

    if row["question_score"] >= 0.40:
        return "Question"

    elif row["agreement_score"] >= row["disagreement_score"] and row["agreement_score"] >= 0.30:
        return "Agreement"

    elif row["disagreement_score"] >= 0.30:
        return "Disagreement"

    else:
        return "Other"

conversation_df["event"] = conversation_df.apply(classify, axis=1)

In [ ]:

ranked_df = conversation_df.sort_values(

    by="highlight_score",

    ascending=False

).reset_index(drop=True)

ranked_df["rank"] = ranked_df.index + 1

ranked_df.head(20)


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,321.68,330.56,05:21,\nPrevious:\nto see that you're doing the tech...,0.55,1.0,0.407942,1.0,0.000000,0.8,0.801981,0.721876,66.66,Question,1
1,836.48,847.48,13:56,\nPrevious:\nuse this checklist. What am I tel...,0.55,0.0,0.613718,1.0,0.666667,0.2,0.846294,0.918223,66.52,Agreement,2
2,229.84,235.20,03:49,"\nPrevious:\nBut you can just aim for a five,\...",0.90,0.0,0.386282,1.0,0.333333,0.4,0.600712,1.000000,63.45,Question,3
3,411.52,416.76,06:51,"\nPrevious:\nAnd let me be clear, by practice,...",0.60,0.0,0.418773,0.0,1.000000,0.6,0.796830,0.778954,62.71,Question,4
4,276.20,278.84,04:36,"\nPrevious:\nhi, come on in. We're having chic...",0.50,1.0,0.462094,1.0,0.666667,0.4,0.604746,0.602221,60.94,Question,5
5,909.48,915.48,15:09,\nPrevious:\nBut think back to when I was demo...,0.60,1.0,0.566787,0.0,0.000000,1.0,0.747692,0.957970,60.16,Question,6
6,460.60,467.20,07:40,\nPrevious:\nGive yourself a chance. Practice ...,0.55,1.0,0.494585,1.0,0.000000,0.4,0.756581,0.970839,59.92,Question,7
7,524.72,528.20,08:44,\nPrevious:\nWhy do I do this to myself? And a...,0.85,1.0,0.801444,0.0,0.333333,1.0,0.616072,0.476131,57.72,Question,8
8,255.08,259.72,04:15,\nPrevious:\nJust make the shape of a confiden...,0.70,1.0,0.870036,1.0,0.000000,0.4,0.787317,0.593504,57.50,Question,9
9,103.04,107.48,01:43,\nPrevious:\nto make you think that? Different...,0.55,1.0,0.418773,0.0,0.333333,0.8,0.630028,0.921648,56.27,Question,10


In [ ]:
import os

os.makedirs("output", exist_ok=True)

ranked_df.to_csv(

    "WEEK1_HIGHLIGHT EXTRACTION.csv",

    index=False

)

print("Saved Successfully!")

Saved Successfully!


In [ ]:
ranked_df[

[
    "rank",
    "timestamp",
    "event",
    "highlight_score",
    "conversation"

]

].head(20)

,rank,timestamp,event,highlight_score,conversation
0,1,05:21,Question,66.66,\nPrevious:\nto see that you're doing the tech...
1,2,13:56,Agreement,66.52,\nPrevious:\nuse this checklist. What am I tel...
2,3,03:49,Question,63.45,"\nPrevious:\nBut you can just aim for a five,\..."
3,4,06:51,Question,62.71,"\nPrevious:\nAnd let me be clear, by practice,..."
4,5,04:36,Question,60.94,"\nPrevious:\nhi, come on in. We're having chic..."
5,6,15:09,Question,60.16,\nPrevious:\nBut think back to when I was demo...
6,7,07:40,Question,59.92,\nPrevious:\nGive yourself a chance. Practice ...
7,8,08:44,Question,57.72,\nPrevious:\nWhy do I do this to myself? And a...
8,9,04:15,Question,57.50,\nPrevious:\nJust make the shape of a confiden...
9,10,01:43,Question,56.27,\nPrevious:\nto make you think that? Different...


In [ ]:
from google.colab import files

files.download("WEEK1_HIGHLIGHT EXTRACTION.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Keep only one highlight within a 20-second window
MIN_GAP = 20

selected = []

for _, row in ranked_df.iterrows():

    current_time = row["start"]

    keep = True

    for previous in selected:

        if abs(current_time - previous["start"]) < MIN_GAP:

            keep = False
            break

    if keep:
        selected.append(row)

In [ ]:
# Create dataframe after removing overlaps
selected_df = pd.DataFrame(selected)

selected_df = selected_df.sort_values(
    "highlight_score",
    ascending=False
)

selected_df = selected_df.reset_index(
    drop=True
)

print("Original highlights:")
print(len(ranked_df))

print("\nFiltered highlights:")
print(len(selected_df))

selected_df.head(20)

Original highlights:
189

Filtered highlights:
35


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,321.68,330.56,05:21,\nPrevious:\nto see that you're doing the tech...,0.55,1.0,0.407942,1.0,0.000000,0.8,0.801981,0.721876,66.66,Question,1
1,836.48,847.48,13:56,\nPrevious:\nuse this checklist. What am I tel...,0.55,0.0,0.613718,1.0,0.666667,0.2,0.846294,0.918223,66.52,Agreement,2
2,229.84,235.20,03:49,"\nPrevious:\nBut you can just aim for a five,\...",0.90,0.0,0.386282,1.0,0.333333,0.4,0.600712,1.000000,63.45,Question,3
3,411.52,416.76,06:51,"\nPrevious:\nAnd let me be clear, by practice,...",0.60,0.0,0.418773,0.0,1.000000,0.6,0.796830,0.778954,62.71,Question,4
4,276.20,278.84,04:36,"\nPrevious:\nhi, come on in. We're having chic...",0.50,1.0,0.462094,1.0,0.666667,0.4,0.604746,0.602221,60.94,Question,5
5,909.48,915.48,15:09,\nPrevious:\nBut think back to when I was demo...,0.60,1.0,0.566787,0.0,0.000000,1.0,0.747692,0.957970,60.16,Question,6
6,460.60,467.20,07:40,\nPrevious:\nGive yourself a chance. Practice ...,0.55,1.0,0.494585,1.0,0.000000,0.4,0.756581,0.970839,59.92,Question,7
7,524.72,528.20,08:44,\nPrevious:\nWhy do I do this to myself? And a...,0.85,1.0,0.801444,0.0,0.333333,1.0,0.616072,0.476131,57.72,Question,8
8,255.08,259.72,04:15,\nPrevious:\nJust make the shape of a confiden...,0.70,1.0,0.870036,1.0,0.000000,0.4,0.787317,0.593504,57.50,Question,9
9,103.04,107.48,01:43,\nPrevious:\nto make you think that? Different...,0.55,1.0,0.418773,0.0,0.333333,0.8,0.630028,0.921648,56.27,Question,10


In [ ]:
print(ranked_df.columns)

Index(['start', 'end', 'timestamp', 'conversation', 'word_count',
       'contains_question', 'duration', 'agreement_score',
       'disagreement_score', 'question_score', 'tfidf_score',
       'sentiment_score', 'highlight_score', 'event', 'rank'],
      dtype='str')


In [ ]:
# Remove overlapping highlights

MIN_GAP = 20  # Minimum gap between highlights in seconds

selected = []

for _, row in ranked_df.iterrows():

    current_time = row["start"]

    keep = True

    for previous in selected:

        # Ignore highlights that are too close together
        if abs(current_time - previous["start"]) < MIN_GAP:
            keep = False
            break

    if keep:
        selected.append(row)

In [ ]:
# Create dataframe after removing overlaps

selected_df = pd.DataFrame(selected)

selected_df = selected_df.sort_values(
    "highlight_score",
    ascending=False
)

selected_df = selected_df.reset_index(
    drop=True
)

print("Original highlights:", len(ranked_df))
print("Filtered highlights:", len(selected_df))

selected_df.head(10)

Original highlights: 189
Filtered highlights: 35


,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,321.68,330.56,05:21,\nPrevious:\nto see that you're doing the tech...,0.55,1.0,0.407942,1.0,0.000000,0.8,0.801981,0.721876,66.66,Question,1
1,836.48,847.48,13:56,\nPrevious:\nuse this checklist. What am I tel...,0.55,0.0,0.613718,1.0,0.666667,0.2,0.846294,0.918223,66.52,Agreement,2
2,229.84,235.20,03:49,"\nPrevious:\nBut you can just aim for a five,\...",0.90,0.0,0.386282,1.0,0.333333,0.4,0.600712,1.000000,63.45,Question,3
3,411.52,416.76,06:51,"\nPrevious:\nAnd let me be clear, by practice,...",0.60,0.0,0.418773,0.0,1.000000,0.6,0.796830,0.778954,62.71,Question,4
4,276.20,278.84,04:36,"\nPrevious:\nhi, come on in. We're having chic...",0.50,1.0,0.462094,1.0,0.666667,0.4,0.604746,0.602221,60.94,Question,5
5,909.48,915.48,15:09,\nPrevious:\nBut think back to when I was demo...,0.60,1.0,0.566787,0.0,0.000000,1.0,0.747692,0.957970,60.16,Question,6
6,460.60,467.20,07:40,\nPrevious:\nGive yourself a chance. Practice ...,0.55,1.0,0.494585,1.0,0.000000,0.4,0.756581,0.970839,59.92,Question,7
7,524.72,528.20,08:44,\nPrevious:\nWhy do I do this to myself? And a...,0.85,1.0,0.801444,0.0,0.333333,1.0,0.616072,0.476131,57.72,Question,8
8,255.08,259.72,04:15,\nPrevious:\nJust make the shape of a confiden...,0.70,1.0,0.870036,1.0,0.000000,0.4,0.787317,0.593504,57.50,Question,9
9,103.04,107.48,01:43,\nPrevious:\nto make you think that? Different...,0.55,1.0,0.418773,0.0,0.333333,0.8,0.630028,0.921648,56.27,Question,10


In [ ]:
selected_df[
    ["start", "event", "highlight_score"]
].head(10)

,start,event,highlight_score
0,321.68,Question,66.66
1,836.48,Agreement,66.52
2,229.84,Question,63.45
3,411.52,Question,62.71
4,276.20,Question,60.94
5,909.48,Question,60.16
6,460.60,Question,59.92
7,524.72,Question,57.72
8,255.08,Question,57.50
9,103.04,Question,56.27


In [ ]:
# Define clip duration settings

BUFFER_BEFORE = 5
BUFFER_AFTER = 5

CLIP_DURATION = (
    BUFFER_BEFORE +
    BUFFER_AFTER
)

MAX_VIDEO_DURATION = 60

In [ ]:
# Calculate maximum number of clips

MAX_CLIPS = (
    MAX_VIDEO_DURATION //
    CLIP_DURATION
)

print("Maximum clips:", MAX_CLIPS)

Maximum clips: 6


In [ ]:
# Select top highlights

final_highlights = selected_df.head(
    MAX_CLIPS
)

final_highlights

,start,end,timestamp,conversation,word_count,contains_question,duration,agreement_score,disagreement_score,question_score,tfidf_score,sentiment_score,highlight_score,event,rank
0,321.68,330.56,05:21,\nPrevious:\nto see that you're doing the tech...,0.55,1.0,0.407942,1.0,0.000000,0.8,0.801981,0.721876,66.66,Question,1
1,836.48,847.48,13:56,\nPrevious:\nuse this checklist. What am I tel...,0.55,0.0,0.613718,1.0,0.666667,0.2,0.846294,0.918223,66.52,Agreement,2
2,229.84,235.20,03:49,"\nPrevious:\nBut you can just aim for a five,\...",0.90,0.0,0.386282,1.0,0.333333,0.4,0.600712,1.000000,63.45,Question,3
3,411.52,416.76,06:51,"\nPrevious:\nAnd let me be clear, by practice,...",0.60,0.0,0.418773,0.0,1.000000,0.6,0.796830,0.778954,62.71,Question,4
4,276.20,278.84,04:36,"\nPrevious:\nhi, come on in. We're having chic...",0.50,1.0,0.462094,1.0,0.666667,0.4,0.604746,0.602221,60.94,Question,5
5,909.48,915.48,15:09,\nPrevious:\nBut think back to when I was demo...,0.60,1.0,0.566787,0.0,0.000000,1.0,0.747692,0.957970,60.16,Question,6


In [ ]:
# Calculate total duration

final_duration = (
    len(final_highlights)
    * CLIP_DURATION
)

print(
    "Final video duration:",
    final_duration,
    "seconds"
)

Final video duration: 60 seconds


In [ ]:
# Save selected highlights for clip extraction

final_highlights.to_csv(
    "final_highlights.csv",
    index=False
)

print(
    "Final highlights selected!"
)

Final highlights selected!


In [ ]:
final_highlights[
    [
        "start",
        "event",
        "highlight_score"
    ]
]

,start,event,highlight_score
0,321.68,Question,66.66
1,836.48,Agreement,66.52
2,229.84,Question,63.45
3,411.52,Question,62.71
4,276.20,Question,60.94
5,909.48,Question,60.16


In [ ]:
import os

print(os.listdir("downloads"))

['video.mp4']


In [ ]:
# Install video processing library
!pip -q install moviepy

In [ ]:
# Import required libraries
from moviepy.editor import VideoFileClip
import os

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



In [ ]:
# Create folder to store extracted clips
os.makedirs(
    "highlight_clips",
    exist_ok=True
)

In [ ]:
# Buffer duration around highlight
BUFFER_BEFORE = 5
BUFFER_AFTER = 5

In [ ]:
# Load downloaded video
video = VideoFileClip(
    "downloads/video.mp4"
)

print(
    "Video duration:",
    round(video.duration,2),
    "seconds"
)

Video duration: 967.46 seconds


In [ ]:
# Extract clips around highlight moments

saved_clips = []

for idx, row in final_highlights.iterrows():

    highlight_time = row["start"]

    start_time = max(
        0,
        highlight_time - BUFFER_BEFORE
    )

    end_time = min(
        video.duration,
        highlight_time + BUFFER_AFTER
    )

    clip = video.subclip(
        start_time,
        end_time
    )

    output_file = (
        f"highlight_clips/clip_{idx}.mp4"
    )

    clip.write_videofile(
        output_file,
        codec="libx264",
        audio_codec="aac",
        logger=None
    )

    saved_clips.append(
        output_file
    )

    print(
        f"Saved: {output_file}"
    )

Saved: highlight_clips/clip_0.mp4
Saved: highlight_clips/clip_1.mp4
Saved: highlight_clips/clip_2.mp4
Saved: highlight_clips/clip_3.mp4
Saved: highlight_clips/clip_4.mp4
Saved: highlight_clips/clip_5.mp4


In [ ]:
len(saved_clips)

6

In [ ]:
# Import functions for merging clips
from moviepy.editor import (
    VideoFileClip,
    concatenate_videoclips
)

In [ ]:
# Load all extracted clips

clips = []

for clip_file in saved_clips:

    clip = VideoFileClip(
        clip_file
    )

    clips.append(clip)

print(
    "Total clips loaded:",
    len(clips)
)

Total clips loaded: 6


In [ ]:
# Merge all highlight clips

final_video = concatenate_videoclips(
    clips,
    method="compose"
)


In [ ]:
# Save final highlight video

final_video.write_videofile(
    "FINAL_HIGHLIGHTS.mp4",
    codec="libx264",
    audio_codec="aac"
)

Moviepy - Building video FINAL_HIGHLIGHTS.mp4.
MoviePy - Writing audio in FINAL_HIGHLIGHTSTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video FINAL_HIGHLIGHTS.mp4



Moviepy - Done !
Moviepy - video ready FINAL_HIGHLIGHTS.mp4


In [ ]:
import os

print(
    "Final video exists:",
    os.path.exists(
        "FINAL_HIGHLIGHTS.mp4"
    )
)

Final video exists: True
